# 🛡️ Module 04: Robust CRNN-PCEN Gunshot Detector

A **Domain-Shift Resilient** gunshot detection training pipeline based on **Forest Acoustic & Chainsaw Detection Research (NYU Bioacoustics / DCASE)**.

## Key Innovations
1. **PCEN (Per-Channel Energy Normalization)**: Replaces static Log-Mel Spectrograms. Automatically normalizes background noise floor and microphone gain variations over time.
2. **Synthetic Microphone Distortion**: Simulates hardware bandpass filtering, clipping, noise injection, and mic gain scaling during training.
3. **Open-Source CRNN (Conv2D + Bi-GRU + Attention)**: Combines spatial PCEN time-frequency pattern extraction with temporal onset sequence modeling.
4. **Strict 1:1 Balanced Evaluation**: Validation and Test sets are forced to a **50/50 balance (1:1 ratio)** to prevent model cheating via majority non-gunshot bias.
5. **Zero Data Leakage**: GroupKFold by original source recording.

In [4]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
# %pip install -q tensorflow librosa soundfile scikit-learn matplotlib seaborn tqdm pandas numpy scipy

In [1]:
# ============================================================
# CELL 2: Imports & Pipeline Setup
# ============================================================
import os
import sys
import re
import json
import random
import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score,
    fbeta_score, roc_curve
)

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# Import PCEN & Mic Pipeline
sys.path.insert(0, str(Path.cwd()))
from pcen_mic_pipeline import compute_pcen, simulate_microphone_effects

print(f'TensorFlow: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('✅ Pipeline modules loaded successfully.')

c:\Users\aadit\.conda\envs\Shooter_model\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow: 2.16.1
GPU Available: False
✅ Pipeline modules loaded successfully.


In [6]:
# ============================================================
# CELL 3: Configuration
# ============================================================
DATA_DIR = Path.cwd().resolve().parent / 'Data' / 'SPLIT_DATASET_750MS'

SAMPLE_RATE = 22050
CLIP_DURATION_MS = 750
TARGET_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)

N_MELS = 64
N_FFT = 512
HOP_LENGTH = 128

BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 0.001

OUTPUT_DIR = Path.cwd().resolve() / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 60)
print('MODULE 04 CONFIGURATION')
print('=' * 60)
print(f'Data Directory   : {DATA_DIR}')
print(f'Sample Rate      : {SAMPLE_RATE:,} Hz')
print(f'N_MELS           : {N_MELS}')
print(f'Output           : {OUTPUT_DIR}')
print('=' * 60)

MODULE 04 CONFIGURATION
Data Directory   : C:\Users\aadit\Desktop\Shoot_Catcher\Data\SPLIT_DATASET_750MS
Sample Rate      : 22,050 Hz
N_MELS           : 64
Output           : C:\Users\aadit\Desktop\Shoot_Catcher\04_Robust_CRNN_PCEN\output


In [7]:
# ============================================================
# CELL 4: Data Loading & Strict 1:1 GroupKFold Split
# ============================================================

def extract_source_group(filename):
    stem = Path(filename).stem
    no_prefix = re.sub(r'^c[01]_\d+_', '', stem)
    no_suffix = re.sub(r'_clip\d+.*$', '', no_prefix)
    no_aug = re.sub(r'_aug_\d+dB$', '', no_suffix)
    return no_aug if no_aug else stem

def is_clean_file(filename):
    return '_aug_' not in Path(filename).stem

def collect_audio_files(data_dir):
    data_dir = Path(data_dir)
    files, labels, groups = [], [], []
    for f in data_dir.rglob('*.wav'):
        name = f.parent.name.lower()
        label = 0 if ('class_0' in name or 'nongunshot' in name) else 1 if ('class_1' in name or 'gunshot' in name) else -1
        if label < 0: continue
        if is_clean_file(f.name):
            files.append(str(f))
            labels.append(label)
            groups.append(extract_source_group(f.name))
    return files, labels, groups

all_files, all_labels, all_groups = collect_audio_files(DATA_DIR)
combined = list(zip(all_files, all_labels, all_groups))
random.shuffle(combined)
all_files, all_labels, all_groups = zip(*combined)
all_files, all_labels, all_groups = list(all_files), list(all_labels), list(all_groups)

# Split GroupKFold into Train / Val / Test
unique_groups = list(set(all_groups))
random.shuffle(unique_groups)
n_test = max(1, int(len(unique_groups) * 0.15))
test_groups = set(unique_groups[:n_test])
tv_groups = set(unique_groups[n_test:])

n_val = max(1, int(len(unique_groups) * 0.15))
val_groups = set(list(tv_groups)[:n_val])
train_groups = tv_groups - val_groups

train_f, train_l = [], []
val_f, val_l = [], []
test_f, test_l = [], []

for f, l, g in zip(all_files, all_labels, all_groups):
    if g in test_groups:
        test_f.append(f); test_l.append(l)
    elif g in val_groups:
        val_f.append(f); val_l.append(l)
    else:
        train_f.append(f); train_l.append(l)

# --- FORCE STRICT 1:1 BALANCED RATIO FOR TEST & VAL SETS ---
def balance_dataset_1to1(files, labels):
    files, labels = np.array(files), np.array(labels)
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]
    min_count = min(len(pos_idx), len(neg_idx))
    selected_pos = np.random.choice(pos_idx, min_count, replace=False)
    selected_neg = np.random.choice(neg_idx, min_count, replace=False)
    final_idx = np.concatenate([selected_pos, selected_neg])
    np.random.shuffle(final_idx)
    return files[final_idx].tolist(), labels[final_idx].tolist()

val_f, val_l = balance_dataset_1to1(val_f, val_l)
test_f, test_l = balance_dataset_1to1(test_f, test_l)

print('SPLIT SUMMARY (Strict 1:1 Balanced Evaluation):')
for name, f_list, l_list in [('TRAIN', train_f, train_l), ('VAL', val_f, val_l), ('TEST', test_f, test_l)]:
    arr = np.array(l_list)
    print(f' {name:5s}: {len(f_list):>6,} files | 🔫 {int(np.sum(arr==1)):>5,} | 🎵 {int(np.sum(arr==0)):>5,}')

SPLIT SUMMARY (Strict 1:1 Balanced Evaluation):
 TRAIN: 31,249 files | 🔫 3,195 | 🎵 28,054
 VAL  :  1,336 files | 🔫   668 | 🎵   668
 TEST :  1,274 files | 🔫   637 | 🎵   637


In [8]:
# ============================================================
# CELL 5: PCEN Extraction + Synthetic Mic Augmentation
# ============================================================
import soundfile as sf

def load_and_extract_pcen(file_list, label_list, sr, target_samples, n_mels, n_fft, hop_length, augment=False):
    X, y = [], []
    target_shape = None
    
    for filepath, label in tqdm(zip(file_list, label_list), total=len(file_list), desc='Extracting PCEN'):
        try:
            audio, _ = sf.read(filepath)
            if audio is None or len(audio) == 0:
                continue
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            audio = np.nan_to_num(audio).astype(np.float32)
            
            if len(audio) < n_fft:
                continue
                
            if len(audio) >= target_samples:
                audio = audio[:target_samples]
            else:
                audio = np.pad(audio, (0, target_samples - len(audio)))
            
            # Peak normalize
            peak = np.max(np.abs(audio))
            if peak > 1e-6:
                audio = audio / peak
            
            # 1. Clean PCEN
            pcen = compute_pcen(audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)
            if pcen is None or pcen.ndim != 2 or pcen.shape[0] != n_mels or pcen.shape[1] < 10:
                continue
                
            if target_shape is None:
                target_shape = pcen.shape
            
            if pcen.shape != target_shape:
                res = np.zeros(target_shape, dtype=np.float32)
                h, w = min(pcen.shape[0], target_shape[0]), min(pcen.shape[1], target_shape[1])
                res[:h, :w] = pcen[:h, :w]
                pcen = res
            
            X.append(pcen)
            y.append(label)
            
            # 2. Synthetic Mic Distorted PCEN (For Training Augmentation)
            if augment:
                mic_audio = simulate_microphone_effects(audio, sr=sr)
                pcen_mic = compute_pcen(mic_audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)
                if pcen_mic is not None and pcen_mic.ndim == 2 and pcen_mic.shape[0] == n_mels and pcen_mic.shape[1] >= 10:
                    if pcen_mic.shape != target_shape:
                        res = np.zeros(target_shape, dtype=np.float32)
                        h, w = min(pcen_mic.shape[0], target_shape[0]), min(pcen_mic.shape[1], target_shape[1])
                        res[:h, :w] = pcen_mic[:h, :w]
                        pcen_mic = res
                    X.append(pcen_mic)
                    y.append(label)
                
        except Exception:
            continue
            
    if len(X) == 0:
        raise ValueError('No valid audio features were extracted!')

    X = np.array(X, dtype=np.float32)[..., np.newaxis]
    y = np.array(y, dtype=np.float32)
    return X, y

# Extract
print('\n📥 Extracting TRAINING PCEN features (Clean + Synthetic Mic)...')
X_train, y_train = load_and_extract_pcen(train_f, train_l, SAMPLE_RATE, TARGET_SAMPLES, N_MELS, N_FFT, HOP_LENGTH, augment=True)

print('\n📥 Extracting VALIDATION PCEN features...')
X_val, y_val = load_and_extract_pcen(val_f, val_l, SAMPLE_RATE, TARGET_SAMPLES, N_MELS, N_FFT, HOP_LENGTH, augment=False)

print('\n📥 Extracting TEST PCEN features...')
X_test, y_test = load_and_extract_pcen(test_f, test_l, SAMPLE_RATE, TARGET_SAMPLES, N_MELS, N_FFT, HOP_LENGTH, augment=False)

# Global Normalization Stats
train_mean = float(np.mean(X_train))
train_std = float(np.std(X_train))

X_train = (X_train - train_mean) / max(train_std, 1e-6)
X_val = (X_val - train_mean) / max(train_std, 1e-6)
X_test = (X_test - train_mean) / max(train_std, 1e-6)

# Save Stats
stats = {'mean': train_mean, 'std': train_std}
(OUTPUT_DIR / 'pcen_stats.json').write_text(json.dumps(stats))

print(f'\n✅ PCEN Normalization Stats: mean={train_mean:.4f}, std={train_std:.4f}')
print(f'X_train shape: {X_train.shape} | X_test shape: {X_test.shape}')


📥 Extracting TRAINING PCEN features (Clean + Synthetic Mic)...


Extracting PCEN:  67%|██████▋   | 21036/31249 [07:15<03:31, 48.27it/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 6: CRNN Architecture (Conv2D + Bi-GRU + Attention)
# ============================================================

def build_robust_crnn(input_shape):
    inputs = layers.Input(shape=input_shape, name='pcen_input')
    
    # 2D Conv Spatial Feature Extraction
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Permute to (Batch, Time_Steps, Mel_Bins, Channels)
    x = layers.Permute((2, 1, 3))(x)
    
    # Reshape for Temporal Sequence Processing: (Batch, time_steps, features)
    feat_dim = (input_shape[0] // 4) * 64
    x = layers.Reshape((-1, feat_dim))(x)
    
    # Bidirectional Recurrent Sequence Modeling (Captures Onset -> Decay)
    x = layers.Bidirectional(layers.GRU(64, return_sequences=True))(x)
    
    # Temporal Pooling / Global Average
    x = layers.GlobalAveragePooling1D()(x)
    
    # Classification Head
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='gunshot_output')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='Robust_CRNN_PCEN')
    return model

model = build_robust_crnn(X_train.shape[1:])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

model.summary()

Model: "Robust_CRNN_PCEN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pcen_input (InputLayer)         │ (None, 64, 130, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 130, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 130, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 65, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 65, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 65, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 32, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 32, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32, 128)        │       418,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gunshot_output (Dense)          │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 446,081 (1.70 MB)

 Trainable params: 445,889 (1.70 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# ============================================================
# CELL 7: Training
# ============================================================
model_path = OUTPUT_DIR / 'crnn_pcen_best.h5'

training_callbacks = [
    callbacks.EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(str(model_path), monitor='val_auc', save_best_only=True, mode='max', verbose=1)
]

print('\n🚀 Training Robust CRNN-PCEN Detector...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=training_callbacks,
    verbose=1
)

# Plot history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['auc'], label='Train AUC')
axes[1].plot(history.history['val_auc'], label='Val AUC')
axes[1].set_title('AUC Score')
axes[1].legend()
plt.savefig(OUTPUT_DIR / 'crnn_training_history.png', dpi=150)
plt.show()

NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
# ============================================================
# CELL 8: 1:1 Strict Balanced Test Evaluation
# ============================================================
model = keras.models.load_model(str(model_path))

print('🔍 Evaluating Robust CRNN-PCEN on 1:1 Strict Test Set...\n')
y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Non-Gunshot', 'Gunshot'], digits=4))

roc_auc = roc_auc_score(y_test, y_pred_prob)
pr_auc = average_precision_score(y_test, y_pred_prob)
f2 = fbeta_score(y_test, y_pred, beta=2)

print(f'\nFINAL METRICS (1:1 Balanced Split):')
print(f'  ROC-AUC : {roc_auc:.4f}')
print(f'  PR-AUC  : {pr_auc:.4f}')
print(f'  F2-Score: {f2:.4f}')

# Plots
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-GS', 'Gunshot'], yticklabels=['Non-GS', 'Gunshot'])
plt.title('1:1 Balanced Confusion Matrix')
plt.savefig(OUTPUT_DIR / 'crnn_confusion_matrix.png', dpi=150)
plt.show()